# CoChem-BENCH Adversarial Audit: Document 91 - PubChem-Database Cache Speeds
**Target**: `cochem_bench.intake.caching`
**Active Extensions**: `/pubchem-database`

1. **The Cache Miss Bomb**: Query PubChem for 100 standard CIDs. The first pass will take network time.
2. **Speed Audit**: Immediately query the exact same 100 CIDs again. `CoChem-BENCH` MUST verify the secondary queries pull strictly from local storage.
3. **Latency Assertion**: Track the `time.perf_counter()`. If the 100 cached retrievals take $> 1.0$ seconds total, the caching implementation fails the efficiency limits.
4. **Validation**: Numerical assertion that pass 2 is at least $100\times$ faster than pass 1.

In [ ]:
import time
import sys
import traceback

def run_audit():
    try:
        from cochem_bench.intake.caching import get_pubchem_records
    except ImportError:
        try:
            from cochem_bench.intake.caching import get_pubchem_records
        except ImportError as e:
            print("[FAIL] The target module 'cochem_bench.intake.caching' (or 'cochem_bench.intake.caching') could not be imported.")
            print(f"Details: {e}")
            sys.exit(1)

    cids = list(range(2244, 2344)) # 100 CIDs

    try:
        # Pass 1: Cache miss
        start_time_1 = time.perf_counter()
        pass1_results = get_pubchem_records(cids)
        end_time_1 = time.perf_counter()
        pass1_duration = end_time_1 - start_time_1
        
        print(f"Pass 1 (Network) Duration: {pass1_duration:.4f} seconds")
        
        # Pass 2: Cache hit
        start_time_2 = time.perf_counter()
        pass2_results = get_pubchem_records(cids)
        end_time_2 = time.perf_counter()
        pass2_duration = end_time_2 - start_time_2
        
        print(f"Pass 2 (Cache) Duration: {pass2_duration:.4f} seconds")
        
        # Latency Assertion
        if pass2_duration >= 1.0:
            print(f"[FAIL] Cache retrieval took >= 1.0 seconds: {pass2_duration:.4f} s")
            sys.exit(1)
        
        speedup = pass1_duration / pass2_duration
        print(f"Speedup: {speedup:.2f}x")
        if pass2_duration * 100 > pass1_duration:
            print(f"[FAIL] Pass 2 is not at least 100x faster than pass 1! Speedup: {speedup:.2f}x")
            sys.exit(1)
            
        print("[PASS] PubChem-Database Cache Speeds audit passed.")
    except Exception as e:
        print(f"[FAIL] Unexpected error during execution: {e}")
        traceback.print_exc()
        sys.exit(1)

run_audit()